In [ ]:
!pip install langchain langchain_ollama langchain-core

In [3]:
from langchain_core.prompts import ChatPromptTemplate as cpt
#from langchain_core.prompts.chat import SystemMessagePromptTemplate as spt,HumanMessagePromptTemplate as hpt
from langchain_ollama import ChatOllama
ollama_model=ChatOllama(model="tinyllama",temperature=0)

In [19]:
#Creating a chatbot for the mentioned travel agent.

sys_msg="""Your name is Bob. You role is to be an expert travel agent. Yo should answer only Travel related questions which can include flight details, accommodation details, visa requirements, travel insurance, destinations and any other travel related tips and queries. If the question is not related to travel, you should politely decline to answer and ask the user to ask a travel related question only and respond exactly with : 'I can not help with it.' Always respond in a friendly tone and should be concise and to the point. Do not exceed the response beyond 30 words."""

#human_msg="Can you please help me with {topic}"
#template=cpt.from_messages([spt.from_template(sys_msg),hpt.from_template(human_msg)])



In [21]:
#Response needs to be streamed(token-by-token) with full context passed to model
from langchain_core.output_parsers import StrOutputParser

# Original system message
original_sys_msg = sys_msg

# List to store conversation history
conversation = []
summarized_history = ""
# conversation loop
while True:
    user_input = input("\nWhat are you looking for ? (type 'exit' to quit): ")

    if user_input.lower() == 'exit':
        break

    # Add user message to the list
    conversation.append({"role": "user", "content": user_input})

    # full context

    chat_template = [
        ("system", original_sys_msg),
        *[(msg["role"], msg["content"]) for msg in conversation], ("human",summarized_history)
    ]

    # Create dynamic template with all messages and stream response
    context_template = cpt.from_messages(chat_template)
    context_chain = context_template | ollama_model | StrOutputParser()

    # Stream the output
    streamed_response = []
    stream_output = context_chain.stream({})
    for st in stream_output:
        print(st, end="", flush=True)
        streamed_response.append(st)
    print()  # New line at the end

    # Full response as a single string
    full_response = "".join(streamed_response)

    # Add response to conversation history
    conversation.append({"role": "assistant", "content": full_response})

    # Check if we've reached 5 exchanges (10 messages total)
    num_exchanges = len(conversation) // 2
    if num_exchanges == 5:
        print("\n" + "="*60)
        print("REACHED 5 CONVERSATIONS - GENERATING SUMMARY")
        print("="*60)
        
        # Format conversation for summary
        conversation_text = "\n".join([
            f"{msg['role'].upper()}: {msg['content']}" 
            for msg in conversation
        ])

        # Create summary prompt
        summary_template = cpt.from_messages([
            ("system", "You are a helpful assistant that summarizes conversations concisely."),
            ("human", f"Please summarize this travel agent conversation in 3-4 bullet points:\n\n{conversation_text}")
        ])
        
        summary_chain = summary_template | ollama_model | StrOutputParser()
        
        print("\nSUMMARY:\n")
        summary_output = summary_chain.invoke({})
        print(summary_output)
        print("\n" + "="*60)
        
        # Update summarized history for next round
        summarized_history += f"\n\nPrevious conversation summary:\n{summary_output}"

        # Reset conversation history but keep the summary in context
        conversation = []

        # Ask if user wants to continue or exit
        continue_choice = input("\nWould you like to continue chatting? (yes/no): ")
        if continue_choice.lower() != 'yes':
            break



Sure, I'd be happy to help you plan for a 3-day trip to Quebec! Here are some suggestions:

1. Start your trip in Montreal - This vibrant city is the capital of Quebec and offers plenty of attractions, including the Old Port district, Place des Arts, and the Montreal Museum of Fine Arts. You can also take a stroll along the St. Lawrence River or visit the Notre-Dame Basilica.

2. Explore Quebec City - This historic city is home to the iconic Château Frontenac, the Parliament Buildings, and the Plains of Abraham. You can also visit the Musée de la Civilisation, which explores the history and culture of Quebec's Indigenous peoples.

3. Take a boat tour along the St. Lawrence River - This is a great way to see the scenic countryside and learn about Quebec's maritime history. You can take a boat tour with a local guide or book a private tour for a more personalized experience.

4. Visit the Montmorency Falls - These waterfalls are located in the Laurentians region of Quebec, and are a must